# Dimension Reduction

## Stochastic Neighbor Embedding (SNE)

The objective of this method is to find an appropriate representation of a set of $n$ data points in dimension $d$ of space $(x_1, x_2, \cdots, x_n) \in \mathbb{R}^{n \times d}$ into a space of dimension $d' \leq d$. This resulting data set is noted $(y_1, y_2, \cdots, y_n) \in \mathbb{R}^{n \times d'}$.

The key idea of t-SNE is to conserve distance between points, independantly of their precise configuration. First, define the conditional probability that particle $j$ is at a given distance of particle $i$:
$$
p_{j|i} = \frac{e^{-||x_i - x_j||^2/(2\sigma_i^2)}}{\sum\limits_{k \neq i}e^{-||x_i - x_k||^2/(2\sigma_i^2)}},
$$
meaning that each particle is the center of a gaussian law of variance $\sigma^2_i$ describing the probability density of its neighbouring points.
Very similarly, now define:
$$
q_{j|i} = \frac{e^{-||y_i - y_j||^2}}{\sum\limits_{k \neq i}e^{-||y_i - y_k||^2}}.
$$

Note that the distance appearing in $p_{j|i}$ are in $\mathbb{R}^d$ whereas the distance is in $\mathbb{R}^{d'}$ for $q_{i|j}$.
We set $\forall i \in \{1, \cdots, d' \}, q_{i|i}=p_{i|i}=0$.

The goal is then to find the points $(y_1, y_2, \cdots, y_n) \in \mathbb{R}^{n \times d'}$ that minimizes the discrepency between $q_{i|j}$ and $p_{i|j}$. We take here a strong metric, namely, the Kullback-Leibler divergence:
$$
C = KL(P||Q) = \sum\limits_{i=1}^n KL(P_i||Q_i) = \sum\limits_{i=1}^n \sum_{j = 1}^n p_{j|i} \log(\frac{p_{j|i}}{q_{j|i}}) = \sum\limits_{i=1}^n H(P_i||Q_i) - H(P_i),
$$
assuming $\frac{p_{i|i}}{q_{i|i}} = 1$. Note that $Q$ is the distribution on the right side of the divergence, meaning that it is equivalent to a Maximum likelihood objective. The optimal point will be find via a first order steepest ascent method. Keeping only terms with $Q_i$ in the KL divergence, one obtains:
$$
KL(P_i, Q_i) \propto -\sum_{j=1}^n p_{j|i}\log(q_{j|i}) = \sum_{j=1}^n p_{j|i} \bigg(||y_i - y_j||^2 + \log\big(\sum\limits_{k\neq i} e^{-||y_k - y_i||^2}\big) \bigg)
$$

A major weakness appears here, the objective function is not convex w.r.t $(y_1, \cdots, y_n)$.

The main remaining question is: how to choose the 'best' $\sigma_i$ for each particle $i$. A small $\sigma_i$ is suited for dense regions, while large $\sigma_i$ is very much relevant for sparse regions.

With this objective in mind, one can define the perplexity of $p_i = p_{\cdot|j}$ as $\text{Perp}(p_i) = 2^{H(p_i)}$, where the entropy $H$ of $p_i$ is $H(p_i) = -\sum\limits_{j=1}^n p_{j|i} \log(p_{j|i})$. Small perplexity put emphasis on local structure, because it requires a small $\sigma_i$. On the opposite, a large perplexity put more emphasis on the global structure.
The operator fixes a perplexity target $P_i$, one per particle $X_i$ in the data set, and SNE performs a binary search (by taking the geometric mean) for the value of $\sigma_i$ that produce this value. It can be done with a binary search precisely because the entropy is an increasing function of $\sigma_i$, one can convince ourself easily with a little bit of physical intuition by interpreting $\sigma_i$ as a temperature. Typical value of particle perplexity should be between $5$ and $50$. Perplexity is a smooth effective measure of the number of neighbours.

To perform gradient ascent, the gradient of the Kullback-Leibler divergence w.r.t $Y_i$ should be computed. It admits a surprising simple expression:
$$
\frac{\partial C}{\partial y_i} = 2\sum\limits_{j=1}^n (p_{j|i} - q_{j|i} + p_{i|j} - q_{i|j})(y_i - y_j)
$$

An updates takes the form, $\forall i \in \{1, \cdots, d'\}$:
$$
\begin{equation}
y_{i}^{(n+1)} =y_i^{(n)} + \eta_n \frac{\partial C}{\partial y_i} + \alpha_n(y_i^{(n)} - y_i^{(n-1)}),
\end{equation}
$$
where $\eta_n$ is the learning rate and $\alpha_n$ is the momentum parameter at iteration $n$.

During the first iterations, a gaussian noise with decreasing variance is added.

In a nutshell, I should make a method to compute $p_{i|j}$, which amounts to compute the distance $||x_i - x_j||_2^2$ for all pairs of points. I should make another method to figure out an appropriate $\sigma_i$ for each of the points. Eventually, a method to run the optimization algorithm is necessary.

[1] https://www.jmlr.org/papers/volume9/vandermaaten08a/vandermaaten08a.pdf

In [4]:
import numpy as np
import matplotlib.pyplot as plt

# def compute_kern_dist(X: np.ndarray, kern: callable):
#     return kern(X, X)

def compute_sq_eucl_dist(X: np.ndarray):
    """
    Compute the squared euclidian distance between all pairs of points.
    """
    return np.sum(X**2, axis=-1)

def compute_pji(X: np.ndarray, sigma: np.ndarray):
    """
    Compute the distribution matrix whose i-th row and j-th column is the probability of having particle j at this distance from particle i.
    Args:
        - X (np.ndarray): Set of data point. It should be of shape (n x d), where n is the number of samples, and d the dimension of space they live in.
        - sigma (np.ndarray): Variange of the gaussian. It should be of shape (n,).
    """
    assert np.all(sigma > 0), "All entry in sigma array should be positive."
    dist = compute_sq_eucl_dist(X[None,...] - X[:,None,...]) #Compute the distance between all pairs of points.
    P = np.exp(-dist/sigma[:, None])
    np.fill_diagonal(P, 0)
    P /= (np.sum(P, axis=-1))[:,None]
    return P

def compute_qji(Y: np.ndarray):
    return compute_pji(Y, sigma=np.ones(len(Y)))

def compute_entropy(P: np.ndarray):
    assert np.abs(np.sum(P) - 1) < 1e-5, f"P should be a list of probabilities, but it sums to {np.sum(P)}."
    P = P[P>0]
    return -np.sum(P*np.log(P), axis = -1)

def sample_init(n: int, d: int, std: float=1e-2):
    """
    Initialize a set of points (y_1, ..., y_n) in the low dimensional space. It is done by sampling from a random variable.
    Args:
        std: standard deviation for the sampling.
        n: number of samples.
        d: dimension space.
    Returns:
        An array of shape (n, d).
    """
    assert std > 0, "The standard deviation should be positive."

    return np.random.randn(n, d)*std

def compute_grad_C(y: np.ndarray, matrix_pji: np.ndarray[np.ndarray], matrix_qji: np.ndarray[np.ndarray]):
    grad_C = np.empty((len(y), 2))
    
    for i in range(len(y)):
        vect_Y = y[i] - y
        p = matrix_pji[i] - matrix_qji[i]
        p += matrix_pji[:,i] - matrix_qji[:,i]
        grad_C[i] = 2*np.sum(p*vect_Y, axis=0)
    return grad_C

def find_sigma(X: np.ndarray, Perp_tar: np.ndarray):
    """Find the variance for a given perplexity target

    Args:
        X (np.ndarray): Dataset. Should be of shape (n x d). n is the number of samples and d the dimension of space.
        Perp (np.ndarray): Target perplexity for each points in the dataset.
    """
    assert len(X) == len(Perp_tar), "X and Perp_tar should be the same length"
    Sigma_BS = []

    for i, (x, perp) in enumerate(zip(X, Perp_tar)):
        entropy = np.infty
        sigma_min = 1e-2
        sigma_max = 1e5
        sigma = np.sqrt(sigma_max*sigma_min)
        dist = compute_sq_eucl_dist(x - X) #Compute the distance between all pairs of points.
        vect = np.exp(-dist/sigma)
        vect[i] = 0
        vect /= np.sum(vect)
        entropy = compute_entropy(vect)
        while np.abs(np.power(2, entropy) - perp) > 1e-5:
            vect = np.exp(-dist/sigma)
            vect[i] = 0
            vect /= np.sum(vect)
            entropy = compute_entropy(vect)
            if np.power(2, entropy) > perp:
                sigma_max = sigma
                sigma = np.sqrt(sigma*sigma_min)
            else:
                sigma_min = sigma
                sigma = np.sqrt(sigma*sigma_max)
        Sigma_BS.append(sigma)
    return Sigma_BS

if __name__ == "__main__":
    A = np.array([[2, 1], [0, 0], [10, -5]])
    sigma = np.array([1, 10, 2])
    P = compute_pji(A, sigma)
    Q = compute_qji(sample_init(3, 2))
    Sigma = find_sigma(A, np.ones(3))
    print(Sigma)

[4.216965034285822, 4.216965034285822, 4.216965034285822]


## t-distributed Stochastinc Neighbor Embedding (t-SNE)

While it is very similary to SNE, it possess two major differences. The first one is the use of symmetrized cost with simpler gradients, and the second one is that probability density are represented by a t-distribution, and not a Gaussian distribution anymore. Symmetrized probability distribution are more robust to outlier

One introduces the symmetrized probability:
$$
p_{ij} = \frac{p_{i|j} + p_{j|i}}{2n}.
$$

One then introduces the symmetrized cost:
$$
KL(P||Q) = \sum\limits_{i=1}^n \sum\limits_{j=1}^n p_{ij} \log(\frac{p_{ij}}{q_{ij}}),
$$
with as before $\forall i \in \{1, \cdots, d'\}, \, p_{ii} = q_{ii} = 0$.

The second changement is the ansatz on $q_{ij}$. The unkowns are still the positions $(y_1, \cdots, y_n) \in \mathbb{R}^{n\times d'}$, but this time, the density family is the Student distribution $\forall i\neq j$:
$$
q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum\limits_{k=1}^n \sum\limits_{l \neq k} (1 + ||y_l - y_k||^2)^{-1}},
$$
so that the renormalization constant implies $\sum\limits_{i, j} q_{ij} = 1$.

The gradient simply becomes:
$$
\frac{\partial C}{\partial y_i} = 4 \sum_{j=1}^n(p_{ij} - q_{ij})(y_i - y_j)(1 + ||y_i - y_j||^2)^{-1},
$$
and the optimization scheme is the same as (1).

The optimization of the t-SNE objective function is much easier than the cost function of SNE. A fixed number of $T = 1000$ timestep is fixed. The momentum term should be $\alpha(t) = .5$ if $t < 250$ and $\alpha(t) = .8$ if $t \geq 250$.


In [ ]:
def compute_symm_pij(X: np.ndarray, sigma):
    """
    Compute the symmetric probability distribution.
    """
    matrix = compute_pji(X, sigma)
    matrix += matrix.T
    matrix /= 1/(2*len(X))
    return matrix

def compute_symm_qij(Y: np.ndarray):
    """
    Compute student's distribution.
    """
    Q = 1/(1 + compute_sq_eucl_dist(Y[None,...] - Y[:,None,...]))
    np.fill_diagonal(Q, 0) #Cancel the diagonal terms
    Q /= np.sum(Q)
    return Q

def compute_grad_C(y: np.ndarray, matrix_pij: np.ndarray[np.ndarray], matrix_qij: np.ndarray[np.ndarray]):
    """
    Compute the gradient of the KL divergence w.r.t matrix_pij and matrix_qij.
    Args:
        y (np.ndarray): data at which compute we compute the gradient.
        matrix_pij (np.ndarray): symmetrized probability matrix for p.
        matrix_qij (np.ndarray): symmetrized probability matrix for q.
    Returns:
        A array of shape (n,2).
    """
    grad_C = np.empty((len(y), 2))
    for i in range(len(y)):
        p = matrix_pij[i]
        q = matrix_qij[i]
        dist = 1/(1 + np.sum((y[i] - y)**2, axis = -1)) #Sum along coordinates
        grad_C[i] = 4*np.sum((p-q)*(y[i] - y)*dist, axis=0) #Sum along samples
    return grad_C

## Non-Negative Matrix Factorization (NMF)

Matrix factorization refers to a set of computational technique to write any matrix $A \in \mathbb{R}^{n \times p}$ as a product of two or more structured matrices. A first advantage is that one can often take advantage of their structure to store them in a most efficient way than simply naively storing all elements of $A$. Indeed, in general, the space complexity of storing a standard matrix scales as $\mathcal{O}(np)$. However, if $A$ writes as $A=HW$, with $H \in \mathbb{R}^{n\times k}$ and $W \in \mathbb{R}^{k \times p}$ and $k$ a **choosen parameter**, then the storage space complexity scales as $\mathcal{O}((n+p)k)$. If $k \ll \min(p, n)$, then the storage of the two matrices $H$ and $W$ instead of the single $A$ becomes much more economical. Another reason is the dimension reduction, since one compresses in some sense the size of the feature space from $p$ elements to only $k$.

In Non-Negative Matrix Factorization, the objective is to find two such matrices $H$ and $W$ with the constraints that all their entries are positive, assuming that all entries of $A$ are positive. If $n$ is the number of samples, and $p$ the number of features, then $H$ is called the basis matrix and $W$ the components matrix.
Noting $||\cdot||$ the Frobenius norm, $H$ and $W$ should be the solutions to this optimization problem:
$$
\begin{array}{cc}
\min\limits_{H \in \mathbb{R}^{n \times k}, W \in \mathbb{R}^{k \times p}} & \frac12||A - HW||^2 \\
\text{s.t} & H\geq 0, W \geq 0
\end{array}
$$

Without further assumptions, the problem above is not convex because its objective function is not convex, making $\textit{de-facto}$ its resolution complex, several solutions may exist. If one imposes an additional constraint: $H H^\top = \text{Id}$, then it is mathematically equivalent to K-means clustering [1], which is known to be NP-Hard.

Of course, many variants exists, including $L^1$ and $L^2$ regularization. The more general regularized optimization problem being:
$$
\begin{array}{cc}
\min\limits_{H \in \mathbb{R}^{n \times k}, W \in \mathbb{R}^{k \times p}} & \frac12||A - HW||^2 + \alpha_1 p ||H||_1 + \beta_1 n ||W||_1 + \frac12\alpha_2 p ||H||^2 + \frac12\beta_2 n ||W||^2,\\
\text{s.t} & H\geq 0, W \geq 0,
\end{array}
$$
with, of course, $||W||_1 = \sum\limits_{i,j} |W_{i,j}|$.
In practice, the first optimization problem is solved via a multiplicative update rule, as published in [2]. It consists in iterating successively:
$$
\begin{array}{cc}
H \leftarrow \frac{H \odot (W^\top V)}{W^\top W H}, & W \leftarrow \frac{W \odot (VH^\top)}{WHH^\top},
\end{array}
$$
where $\odot$ denotes the Hadamard product and the division is entry-wise.

Another possibility is to minimize the KL divergence between $A$ and $WH$. The formulation is:
$$
\begin{array}{cc}
\min\limits_{H \in \mathbb{R}^{n \times k}, W \in \mathbb{R}^{k \times p}} & D_{KL}(A||WH) \\
\text{s.t} & H\geq 0, W \geq 0
\end{array}
$$
The multiplicative update rule takes the form:
$$
\begin{array}{cc}
H_{a, \mu} \leftarrow H_{a,\mu} \displaystyle\frac{\sum\limits_{i=1}^n W_{i,a} A_{i,\mu}/(WH)_{i,\mu}}{\sum\limits_{k=1}^n W_{ka}} , & W_{i,a} \leftarrow W_{i,a}\displaystyle\frac{\sum\limits_{\mu=1}^p H_{a, \mu} A_{i,\mu}/(WH)_{i,\mu}}{\sum\limits_{\nu = 1}^p H_{a, \nu}},
\end{array}
$$
or otherwise written:
$$
H \leftarrow H \odot \frac{W^\top (A/WH)}{\mathbb{1}^\top W}, W \leftarrow W \odot \frac{H (A/WH)^\top}{H \mathbb{1}},
$$
where $\mathbb{1}$ is a vector filled with ones.

The KL divergence admits the following expression for vectors whose entries do not sum up to $1$: $D_{KL}(U||V) = \sum\limits_{i=1}^n \log(\frac{u_i}{v_i})u_i + v_i - u_i$

[1]. https://ranger.uta.edu/~chqding/papers/NMF-SDM2005.pdf

[2]. https://proceedings.neurips.cc/paper_files/paper/2000/file/f9d1152547c0bde01830b7e8bd60024c-Paper.pdf

In [17]:
import numpy as np
def NMF_norm(X: np.ndarray, n_components: int=2, max_iter: int=300, tol: float=1e-4):
    """
    Compute a Non-Negative Matrix Factorization of X. No regularization is added to the objective function.
    Args:
        X (np.ndarray). Non-Negative Data set. Each row is a data, and each column represents a feature.
        n_components (int). Number of components in the factorized matrices.
        max_iter (int). Maximum number of iterations.
        tol (float). Criteria to if the reconstruction is a good approximation of X.
    Returns:
        Two Non-negative numpy arrays H and W.
    """

    if not isinstance(X, np.ndarray):
        X = np.array(X)
    assert np.all(X >= 0), 'All entries of X should be non-negative'

    n_sample, n_feature = X.shape

    W = np.random.rand(n_sample, n_components)
    H = np.random.rand(n_components, n_feature)
    
    counter=0
    while counter <= max_iter or np.linalg.norm(X - W@H)>tol:
        counter += 1
        
        #1st rule
        WH = W.T @ W @ H
        H *= W.T @ X / (WH)
 
        #2nd rule
        WHT = W @ H @ H.T
        W *= (X@H.T)/WHT
    return W, H

def NMF_KL(X: np.ndarray, n_components: int=2, max_iter: int=50, tol: float=1e-4):
    """
    Compute a Non-Negative Matrix Factorization of X. No regularization is added to the objective function. The approximation is on the sense of the
    KL divergence.
    Args:
        X (np.ndarray). Non-Negative Data set. Each row is a data, and each column represents a feature.
        n_components (int). Number of components in the factorized matrices.
        max_iter (int). Maximum number of iterations.
        tol (float). Criteria to if the reconstruction is a good approximation of X.
    Returns:
        Two Non-negative numpy arrays H and W.
    """

    if not isinstance(X, np.ndarray):
        X = np.array(X)
    assert np.all(X >= 0), 'All entries of X should be non-negative'
    n_sample, n_feature = X.shape
    
    W = np.random.rand(n_sample, n_components)
    H = np.random.rand(n_components, n_feature)
    
    counter=0
    while counter <= max_iter or np.sum(np.log(X/(W@H))*X - X + W@H)>tol:
        counter += 1

        #1st rule
        X_norm = X/(W@H)
        H *= W.T @ X_norm / np.sum(W, axis=0)

        #2nd rule
        X_norm = X/(W@H)
        W *= X_norm @ H.T / np.sum(H, axis=1)
    return W, H

if __name__=='__main__':
    import numpy as np
    X = np.array([[1, 1], [2, 1], [3, 1.2], [4, 1], [5, 0.8], [6, 1]])
    from sklearn.decomposition import NMF
    model = NMF(n_components=2, init='random', random_state=0)
    W = model.fit_transform(X)
    H = model.components_
    print(np.linalg.norm(X - W@H))
    W, H = NMF_norm(X, n_components=2, max_iter=300)
    print(np.linalg.norm(X - W@H))
    W, H = NMF_KL(X, n_components=2, max_iter=256)
    print(np.sum(np.log(X/(W@H))*X - X + W@H))

0.0011599349216014024
9.967863800761315e-05
1.0834440443696636e-06


## Kernel Canonical Correlation Analysis

Take two multivariates random variables $X \in \mathbb{R}^n$ and $Y \in \mathbb{R}^m$. The objective of Canonical Correlation Analysis is to find the directions in $\mathbb{R}^n$ and $\mathbb{R}^m$ onto which $X$ and $Y$ are respectively projected while keeping as much variance as possible. Mathematically, it formulates as:
$$
\max\limits_{\substack{w_X \in \mathbb{R}^n, w_Y \in \mathbb{R}^m \\ w_X, w_Y \neq 0}} \frac{\text{Cov}(w_X^\top X, w_Y^\top Y)}{\sqrt{\mathbb{V}[w_X ^\top X]\mathbb{V}[w_Y^\top Y]}}
$$
For two real-valued univariate random variables U and V, the covariance operator is classically: $\text{Cov}(U, V) = \mathbb{E}[UV] - \mathbb{E}[U] \mathbb{E}[V]$.
Of course, it is not practicable to maximize such a quantity, because not only do we not know the probability measures $\mathbb{P}_X$, $\mathbb{P}_Y$ and $\mathbb{P}_{X,Y}$, but it would amount to maximise a parametric integral, which is far from trivial. Consequently, one solves its empirical counterpart. Take $(X, Y) = (x_i, y_i)_{i \in \{1, \cdots, N\}} \sim \mathbb{P}_{X, Y}^{\otimes N}$ and let:
$$
\max\limits_{\substack{w_X \in \mathbb{R}^n, w_Y \in \mathbb{R}^m \\ w_X, w_Y \neq 0}} \frac{\widehat{\text{Cov}}^N(w_X^\top X, w_Y^\top Y)}{\sqrt{\hat{\mathbb{V}}^N[w_X^\top X] \hat{\mathbb{V}}^N[w_Y^\top Y]}},
$$
where one defines $\widehat{\text{Cov}}^N(U, V) = \frac 1N\sum\limits_{i=1}^NU_iV_i - \frac 1{N^2} \sum\limits_{i=1}^NU_i \sum\limits_{i=1}^NV_i$ with $(U_i, V_i)_{i \in \{1, \dots, N\}} \sim \mathbb{P}_{(U, V)}^{\otimes N}$ $\text{i.i.d}$ samples from the joint law of monovariate random variables $U$ and $V$, and $\hat{\mathbb{V}}^N[U] = \frac 1n \sum\limits_{i=1}^N U_i^2 - \frac{1}{N^2} (\sum\limits_{i=1}^N U_i)^2$.

As it often happens in Machine Learning when dealing with linear methods, after a few manipulations, the problem reduces to an eigenvalue problem, here generalized. This simple observation directly implies that the temporal complexity scales as $\mathcal{O}(N^3)$, which is quite bad. Writing $\Sigma_{X, Y} = \mathbb{E}_{X, Y}[X Y^\top] - \mathbb{E}_X[X]\mathbb{E}_Y[Y]^\top$, $\Sigma_X = \mathbb{E}_{X}[XX^\top] - \mathbb{E}_{X}[X]\mathbb{E}_{X}[X]^\top$, $\Sigma_Y = \mathbb{E}_{Y}[YY^\top] - \mathbb{E}_{Y}[Y]\mathbb{E}_{Y}[Y]^\top $ the covariance and variance matrices, and their (biased) respective estimators: $\hat{\Sigma}_{X, Y}^N = \frac 1N \sum\limits_{i=1}^{N}X_iY_i^\top - \frac 1{N^2} (\sum\limits_{i=1}^{N}X_i) (\sum\limits_{i=1}^{N}Y_i)^\top \in \mathbb{R}^{n \times m}$, $\hat{\Sigma}_X^N = \hat{\Sigma}_{X, X}^N \in \mathbb{R}^{n \times n}, \hat{\Sigma}_Y^N = \hat{\Sigma}_{Y, Y}^N \in \mathbb{R}^{m \times m}$. With this quantities, the original problem re-writes:
$$
\max\limits_{\substack{w_X \in \mathbb{R}^n, w_Y \in \mathbb{R}^m \\ w_X \neq 0, w_Y \neq 0}} \frac{w_X ^\top \Sigma_{X, Y} w_Y}{\sqrt{w_X^\top \Sigma_X w_X w_Y^\top \Sigma_Y w_Y}},
$$

and the empirical one:
$$
\max\limits_{\substack{w_X \in \mathbb{R}^n, w_Y \in \mathbb{R}^m \\ w_X \neq 0, w_Y \neq 0}} \frac{w_X ^\top \hat{\Sigma}_{X, Y}^N w_Y}{\sqrt{w_X^\top \hat{\Sigma}_X^N w_X w_Y^\top \hat{\Sigma}_Y^N w_Y}},
$$

or equivalently,
$$
\begin{array}{cc}
\max\limits_{\substack{w_X \in \mathbb{R}^n, w_Y \in \mathbb{R}^m \\ w_X, w_Y \neq 0}} & w_X ^\top \hat{\Sigma}_{X, Y}^N w_Y \\
\text{s.t} & w_X^\top \hat{\Sigma}_X^N w_X = 1, w_Y^\top \hat{\Sigma}_Y^N w_Y = 1
\end{array}
$$

Now, introduce the lagrangian: $\mathcal{L}(w_X, w_Y, \lambda_X, \lambda_Y) = w_X^\top \hat{\Sigma}^N_{X, Y} w_Y + \lambda_X (w_X^\top \hat{\Sigma}^N_X w_X - 1) + \lambda_Y (w_Y^\top \hat{\Sigma}^N_Y w_Y - 1)$, where $\lambda_X, \lambda_Y \in \mathbb{R}$ are Lagrangian multipliers for the normalization constraints. Its derivatives with respect to $\lambda_X$ and $\lambda_Y$ gives back the normalization constraints, while derivatives with respect to $w_X$ and $w_Y$ yields:
$$
\frac{\partial \mathcal{L}}{\partial w_X} = \hat{\Sigma}^N_{X, Y} w_Y + 2\lambda_X\hat{\Sigma}^N_X w_X = 0,
$$
$$
\frac{\partial \mathcal{L}}{\partial w_Y} = \hat{\Sigma}^N_{X, Y} w_X + 2 \lambda_Y\hat{\Sigma}^N_Y w_Y = 0.
$$
Assuming $\Sigma_{X, Y}$ invertible, one immediately gets the two following equations for $\lambda_X$ and $\lambda_Y$:
$$
\lambda_X\Sigma_X w_X = -\frac{1}{2}\hat{\Sigma}^N_{X, Y} w_Y, \quad \lambda_Y\hat{\Sigma}^N_Y w_Y = -\frac{1}{2}\hat{\Sigma}^N_{X, Y} w_X.
$$

Putting those in the constraints equations and assuming $\lambda_X, \lambda_Y \neq 0$ gives:
$$
-\frac 1{2\lambda_X} w_X^\top \hat{\Sigma}^N_{X, Y} w_Y - 1 = 0 \iff  w_X^\top \hat{\Sigma}^N_{X, Y} w_Y = -2\lambda_X, \quad -\frac 1{2\lambda_Y} w_Y^\top \hat{\Sigma}^N_{X, Y} w_X - 1 = 0 \iff w_Y^\top \hat{\Sigma}^N_{X, Y} w_X = -2\lambda_Y
$$
implying that $\lambda_X = \lambda_Y$ given that $\hat{\Sigma}^N_{X, Y}$ is symmetric. Noting $w = [w_X, w_Y]^\top$ and $\lambda = -2\lambda_X = -2\lambda_Y$, the KKT conditions can be rewritten 
$$
\underbrace{\begin{bmatrix} 0 & \hat{\Sigma}^N_{X, Y} \\ \hat{\Sigma}^N_{X, Y} & 0\end{bmatrix}}_{\Sigma_A \in \mathbb{R}^{n \times n}} = \lambda \underbrace{\begin{bmatrix} \hat{\Sigma}^N_X & 0 \\ 0 & \hat{\Sigma}^N_Y \end{bmatrix}}_{\Sigma_B \in \mathbb{R}^{m\times m}}w.
$$
Assuming $\Sigma_X$ and $\Sigma_Y$ are invertible, it can be re-written as:
$$
\Sigma_B^{-1/2} \Sigma_A \Sigma_B^{-1/2} (\Sigma_B^{1/2}w) = \lambda \Sigma_B^{1/2}w.
$$
Note that $\lambda \geq 0$ because $\Sigma_B^{-1/2} \Sigma_A \Sigma_B^{-1/2}$ is semi-definite positive.

As expected, CCA is related to a (generalized) eigenvalue problem, which can be solved via numerous numerical solvers.

### Kernel CCA

Take again $X$ and $Y$, two random variables taking value in $\mathcal{X}$ and $\mathcal{Y}$ respectively. Take two p.d kernels $K_1: \mathcal{X} \times \mathcal{X} \rightarrow \mathbb{R}$, $K_2: \mathcal{Y} \times \mathcal{Y} \rightarrow \mathbb{R}$, and note $\mathcal{H}_X, \mathcal{H}_Y$ their associated RKHS.  In its framework, KCCA admits the formulation:
$$
\max\limits_{\substack{f \in \mathcal{H}_X, g \in \mathcal{H}_Y \\ f,\, g \neq 0}} \frac{\text{Cov}(f(X), g(Y))}{\sqrt{\mathbb{V}[f(X)]\mathbb{V}[g(Y)]}}.
$$

For the exact same reasons as previously, in practice, one can solely solve its empirical counterpart. Take $(X, Y) = (x_i, y_i)_{i \in \{1, \cdots, n\}} \sim \mathbb{P}_{X, Y}^{\otimes n}$ and let:
$$
\max\limits_{\substack{f \in \mathcal{H}_X, g \in \mathcal{H}_Y \\ f,\, g \neq 0}} \frac{\widehat{\text{Cov}}(f(X), g(Y))}{\sqrt{\hat{\mathbb{V}}[f(X)]\hat{\mathbb{V}}[g(Y)]}}.
$$
Applying the representer theorem (applicability remains to be checked), solutions $f, g$ of the above optimization problem admits the decomposition $\forall x \in \mathcal{X}, f(x) = \sum\limits_{i = 1}^n \alpha_i K_1(x, x_i)$ and $\forall y \in \mathcal{Y}, g(y) = \sum\limits_{i = 1}^n \beta_i K_2(y, y_i)$. Plugging it into the objective function, it follows:
$$
\max\limits_{\substack{\alpha, \beta \in \mathbb{R}^n \\ \alpha,\, \beta \neq 0}} \frac{\alpha^\top K_\alpha K_\beta \beta}{\sqrt{\alpha^\top K_\alpha^2 \alpha \beta^\top K_\beta^2 \beta}},
$$
where $\forall (i, j) \in \{1, \cdots, n\}^2, \, (K_\alpha)_{i, j} = K_1(x_i, x_j) - \frac 1n \sum\limits_{l=1}^{n}K_1(x_i, x_l) - \frac 1n \sum\limits_{m=1}^{n}K_1(x_m, x_j) + \frac{1}{n^2}\sum\limits_{l, m=1}^{n}K_1(x_m, x_l)$ and $\forall (i, j) \in \{1, \cdots, n\}^2, \, (K_\beta)_{i, j} = K_2(y_i, y_j) - \frac 1n \sum\limits_{l=1}^{n}K_2(y_i, y_l) - \frac 1n \sum\limits_{m=1}^{n}K_2(y_m, y_j) + \frac{1}{n^2}\sum\limits_{l, m=1}^{n}K_2(y_l, y_m)$. It can be re-written as a constrained problem:
$$
\begin{array}{cc}
\max\limits_{\substack{\alpha, \beta \in \mathbb{R}^n \\ \alpha,\, \beta \neq 0}} & \alpha^\top K_\alpha K_\beta \beta\\
\text{s.t} & \alpha^\top K_\alpha^2 \alpha = 1, \, \beta^\top K_\beta^2 \beta = 1\\
\end{array}
$$

Very similarly to what was done previously, it also leads to a generalized eigenvalue problem.

Note: The above problem can be easily generalized where we keep not one, but the top $k$ eigenvalues of the problem. For example, linear CCA would admits the form:

$$
\begin{array}{ccc}
\max\limits_{\substack{W_X \in \mathbb{R}^{k\times n}, W_Y \in \mathbb{R}^{k \times m} \\ W_X \neq 0, W_Y \neq 0}} & W_X \Sigma_{X, Y} W_Y^\top \\
\text{s.t} & (W_X)_i^\top \hat{\Sigma}^N_X (W_X)_i = 1, \forall i \in \{1, \cdots, k\} \, & \text{Normalization constraints}\\
 &  (W_Y)_i^\top \hat{\Sigma}^N_Y (W_Y)_i = 1, \forall i \in \{1, \cdots, k\} \\
 & (W_X)_i^\top (W_X)_j = 0 \, \forall i \neq j \in \{1, \cdots, k\} \, &\text{Orthogonality of eigenvectors}\\
 & (W_Y)_i^\top (W_Y)_j = 0 \, \forall i \neq j \in \{1, \cdots, k\}
\end{array}
$$


[1] https://www.jmlr.org/papers/volume8/fukumizu07a/fukumizu07a.pdf

In [ ]:
import numpy as np
from collections.abc import Callable
from scipy.linalg import eig

def RBF(sigma: float = 1):
    def kernel(x: np.ndarray, y: np.ndarray):
        expo = -(np.sum((x[:,None,:] - y[None,:,:])**2, axis = -1))/(2*sigma) #Sum along the data dimension
        return np.exp(expo)
    return kernel

def Linear():
    def kernel(x: np.ndarray, y: np.ndarray):
        assert len(x.shape) == 2 and len(y.shape) == 2, "x and y should have two axis."
        if isinstance(x, list):
            x = np.array(x)
        if isinstance(y, list):
            y = np.array(y)
        return np.einsum('ij, ji -> i', x, y.T)
    return kernel

def KCCA(object):
    def __init__(self, kernel: Callable, n_components: int):
        self.n_components = n_components
        self.kernel = kernel

    def distance(self, X: np.ndarray, Y: np.ndarray):
        """
        Compute the distance between X and Y induced by the kernel. It should be vectorized along the batch axis.
        """
        return self.kernel(X, X) + self.kernel(Y, Y) - 2*self.kernel(X, Y)
    
    def fit(self, X: np.ndarray, Y: np.ndarray):
        """
        Find the directions that correlate the most X and Y.
        Args:
            X (np.ndarray). First dataset. Should be of size (batch, n).
            Y (np.ndarray). Second dataset. Should be of size (batch, m).
        """
        N, n = X.shape
        N, m = Y.shape
        Sigma_X = X.T @ X/N #Empirical estimation of covariance matrix
        Sigma_Y = Y.T @ Y/N
        
        Sigma_XY = X.T @ Y/N
        
        inv_Sigma_X = np.linalg.inv(Sigma_X)
        inv_Sigma_Y = np.linalg.inv(Sigma_Y)

        Sigma_A = np.block([[np.zeros((m, n)), Sigma_XY], [Sigma_XY, np.zeros((m, n))]])
        inv_Sigma_B = np.block([[inv_Sigma_X, np.zeros((n, m))], [np.zeros((m, n)), inv_Sigma_Y]])
        Sigma = inv_Sigma_B @ Sigma_A
        _, eig_vectors = np.linalg.eig(Sigma)
        self.eig_vectors_X = eig_vectors[:n, :self.n_components]
        self.eig_vectors_Y = eig_vectors[:n, :self.n_components]
    
    def transform(self, X: np.ndarray, Y: np.ndarray=None):
        pass